In [1]:
# =============================================================================
# CAUSAL FORESTS: CONCEPT NOTES
# =============================================================================
#
# WHAT PROBLEM DOES IT SOLVE?
#   Every method so far (TWFE, DiD, Panel Regression) gives you ONE number:
#   the average treatment effect across all units.
#   That average hides the fact that different stores may respond very differently.
#   Causal Forests estimate a SEPARATE treatment effect for every single unit.
#
#   ATE  = Average Treatment Effect         => what TWFE gives you
#   CATE = Conditional Average Treatment Effect => what Causal Forests give you
#   "Conditional" means: conditional on the features of that specific unit.
#
# -----------------------------------------------------------------------------
# THE CORE QUESTION IT ANSWERS
# -----------------------------------------------------------------------------
#   Not just "did the algorithm work?" (TWFE answers that)
#   But "for WHICH stores did it work, and what made them different?"
#   e.g. Large city stores got +$900, small suburban stores got +$100.
#   TWFE would average this into +$500 and you would never know.
#
# -----------------------------------------------------------------------------
# HOW IT WORKS (plain english)
# -----------------------------------------------------------------------------
#   Step 1: Build many decision trees (like a regular Random Forest).
#   Step 2: But instead of splitting to minimize prediction error,
#           each tree splits to SEPARATE units with different treatment effects.
#           It looks for features where: "above this threshold, effect is big.
#           below this threshold, effect is small."
#   Step 3: Within each leaf of the tree, estimate the local treatment effect
#           as the difference in outcomes between treated and control units
#           that landed in that leaf.
#   Step 4: Average across all trees to get a stable estimate per unit.
#
# -----------------------------------------------------------------------------
# HONEST SPLITTING (the key innovation, prevents overfitting)
# -----------------------------------------------------------------------------
#   THE PROBLEM:
#     If you use the same data to both FIND the splits AND ESTIMATE the effect,
#     the tree finds splits that made the effect look big in this specific sample.
#     Then you measure that same lucky fluctuation again when estimating.
#     Result: inflated, overfit estimates that won't hold on new data.
#
#   THE FIX (honesty):
#     Split your data into two completely separate groups BEFORE doing anything.
#       Splitting sample  => used ONLY to decide the tree structure
#                            (where to split, on what feature, at what threshold)
#       Estimation sample => used ONLY to estimate the effect at each leaf
#                            (never influenced the tree structure)
#     Because the estimation sample had no role in choosing the splits,
#     it cannot have inflated those estimates. The estimate is clean.
#
#   THE ANALOGY:
#     Imagine you are a teacher grading your own exam.
#     You write the questions, you know the answers, and you also grade the papers.
#     You will give yourself a perfect score. But that score means nothing
#     because you used the same information to both set up the test and evaluate it.
#     A decision tree does the same thing if you let it.
#
#   THE FIX (honesty):
#     Take your 1000 stores and immediately split them into two separate groups
#     before doing anything else.
#       Splitting sample  (600 stores): used ONLY to decide the tree structure.
#                          Where to split? On what feature? At what threshold?
#                          This sample finds that "foot traffic above 500" is
#                          where heterogeneity lives. Tree structure gets locked in.
#                          This sample is now put away and never touched again.
#
#       Estimation sample (400 stores): comes in fresh after the structure is locked.
#                          These stores never influenced any split decision.
#                          They simply fall down the already-decided tree,
#                          land in their leaf, and the treatment effect is estimated
#                          purely from these fresh observations.
#
#     Because the estimation sample had no role in choosing the splits,
#     it cannot have inflated those estimates. The split was chosen independently.
#     The estimate is clean.
#
# -----------------------------------------------------------------------------
# WHAT YOU GET OUT
# -----------------------------------------------------------------------------
#   tau_hat[i]  = estimated treatment effect for unit i specifically
#   std_err[i]  = uncertainty around that estimate
#   p_value[i]  = is this unit's effect significantly different from zero?
#   variable_importance = which features most drive the heterogeneity in effects?
#
# -----------------------------------------------------------------------------
# WHEN TO USE CAUSAL FORESTS vs PANEL REGRESSION
# -----------------------------------------------------------------------------
#   Use PANEL REGRESSION (TWFE) when:
#     - You want a clean, defensible average causal effect
#     - You have panel data (same units observed over multiple time periods)
#     - Treatment was selective and you need within-unit variation to identify effect
#
#   Use CAUSAL FORESTS when:
#     - You believe the effect varies across units and want to understand who responds
#     - You want to build targeting rules (treat only units where effect > cost)
#     - You have rich feature data on each unit
#     - Your primary goal is personalization, not just "did it work on average"
#
# -----------------------------------------------------------------------------
# THE KEY LIMITATION
# -----------------------------------------------------------------------------
#   Causal Forests require OVERLAP:
#   For every combination of features, you need BOTH treated AND untreated units.
#   If only big stores were treated, the forest cannot estimate effects for small
#   stores because there is no untreated big store to compare against.
#   Panel FE handles selective treatment more gracefully via within-unit over-time
#   comparison. Causal Forests rely on cross-sectional treated vs control comparison.
#
# -----------------------------------------------------------------------------
# THE LIBRARY: econml (by Microsoft Research)
# -----------------------------------------------------------------------------
#   pip install econml
#
#   from econml.dml import CausalForestDML
#   from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
#
#   model = CausalForestDML(
#       model_y      = RandomForestRegressor(),  # removes effect of controls on outcome
#       model_t      = RandomForestClassifier(), # removes effect of controls on treatment
#       n_estimators = 200,
#       honest       = True,                     # turns on honest splitting
#       random_state = 42
#   )
#
#   model.fit(Y=revenue, T=treatment, X=store_features, W=controls)
#
#   tau_hat        = model.effect(store_features)      # per-unit treatment effects
#   ate, lb, ub    = model.ate_interval(store_features) # average effect with CI
#
#   NOTE: CausalForestDML uses Double Machine Learning (DML) under the hood.
#   model_y and model_t first partial out the influence of control variables
#   from both the outcome and the treatment before the causal forest runs.
#   This makes it robust to having many control variables.
#   You can plug in any sklearn-compatible model for model_y and model_t.
#
# =============================================================================